# Train the license plate detector

This notebook is designed to run from GitHub in Google Colab. It clones the repository when needed, mounts Drive for data and checkpoints, requires a GPU, validates the dataset path, and trains a small YOLO detector.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TrayMachi/indonesia-license-plate-model.git"
REPO_DIR = Path("/content/indonesia-license-plate-model")
if not (REPO_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%pip install -q -r /content/indonesia-license-plate-model/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable Runtime > Change runtime type > T4 GPU before training.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/indonesia-license-plate-model/dataset_yolo")
DATASET_YAML = REPO_DIR / "configs" / "dataset.yaml"
DRIVE_ROOT = Path("/content/drive/MyDrive/indonesia-license-plate-model")
RUNS_DIR = DRIVE_ROOT / "runs"
RUN_NAME = "plate-detector"

required = [DATASET_DIR / "images" / "train", DATASET_DIR / "images" / "val", DATASET_DIR / "labels" / "train", DATASET_DIR / "labels" / "val"]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset folders:\n" + "\n".join(missing))
print("Dataset:", DATASET_DIR)
print("Config:", DATASET_YAML)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
train_results = model.train(
    data=str(DATASET_YAML),
    epochs=50,
    imgsz=640,
    batch=-1,
    patience=15,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    cache=False,
)

best_checkpoint = Path(train_results.save_dir) / "weights" / "best.pt"
if not best_checkpoint.exists():
    raise FileNotFoundError(f"Training finished but {best_checkpoint} was not created")
print("Best checkpoint:", best_checkpoint)

In [ ]:
metrics = model.val(data=str(DATASET_YAML), split="val", imgsz=640, device=0)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Artifacts: {Path(train_results.save_dir)}")

Before a serious run:

Review the class definition and dataset balance. Change epochs, imgsz, and RUN_NAME above as needed. The checkpoint, plots, and validation results are written to Drive under the project runs folder.